# "A note on Cross Entropy loss with PyTorch"

- toc: true
- badges: true
- comments: true
- categories: [fastai, pytorch, jupyter]

# Maximizing the likelihood of the labels of the data

Suppose we have $N$ training examples and we have a multiclass problem with $K$ classes. Let $C(i) \in \{1,\ldots, K\}$ be the correct class for the $i$-th training example and $o^{[C(i)]}_{i}$ gives the probability assigned by our classifier to the correct class for the $i$-th training example. We wish that our classifier maximize: $$\prod_{i=1}^{N} o^{[C(i)]}_{i}$$ 

This is equivalent to maximizing $$ln(\prod_{i=1}^{N}o^{[C(i)]}_{i}) = \sum_{i=1}^{N}ln(o^{[C(i)]}_{i})$$ 

Which is the same as minimizing the sum of the negative log likelihoods $$-\sum_{i=1}^{N}ln(o^{[C(i)]}_{i})$$

The above can now serve as a loss function.

Recall Cross Entropy = $-\sum_{k=1}^{K}y^{[k]}ln(o^{[k]})$ where $y$ is the reference distribution over $K$ classes while our predictions over the $K$ classes is given by $o$. Observe this becomes just a single term when, in the reference distribution $y$, only one of the classes has a probability of $1$.

Thus, $-\sum_{i=1}^{N}ln(o^{[C(i)]}_{i})$ can be interpreted as the sum of cross entropy losses across all examples.

In [1]:
from fastai.vision.all import *

Pretend the following are the activations of each class of a multiclass classification problem. So we have 6 examples and in each row we have the activation for each class the example could belong to.

In [2]:
activations = torch.randn((6,2))*2
activations

tensor([[-0.6562, -1.8913],
        [-0.2568,  0.1487],
        [-1.1133, -1.0681],
        [-0.4050,  2.6902],
        [-2.4302, -2.3022],
        [ 0.7186,  3.7159]])

Suppose the correct class of each example is as follows

In [3]:
targets = tensor([0,1,0,1,1,0])
targets

tensor([0, 1, 0, 1, 1, 0])

Take the softmax of the activations

In [4]:
sm_acts = torch.softmax(activations, dim=1)
sm_acts

tensor([[0.7747, 0.2253],
        [0.4000, 0.6000],
        [0.4887, 0.5113],
        [0.0433, 0.9567],
        [0.4680, 0.5320],
        [0.0475, 0.9525]])

Extract the probabilities predicted for the correct class.

In [5]:
idx = range(6)
list(idx)

[0, 1, 2, 3, 4, 5]

In [6]:
p_correct_class = sm_acts[idx, targets]
p_correct_class

tensor([0.7747, 0.6000, 0.4887, 0.9567, 0.5320, 0.0475])

Take the log of the softmax activations

In [7]:
torch.log(sm_acts)

tensor([[-0.2553, -1.4904],
        [-0.9163, -0.5108],
        [-0.7160, -0.6708],
        [-3.1395, -0.0443],
        [-0.7592, -0.6312],
        [-3.0460, -0.0487]])

Computing the softmax of the activations and then taking the log is equivalent to applying PyTorch's log_softmax function directly to the original activations. We want to do the latter because it will faster and more accurate.

In [8]:
torch.log_softmax(activations, dim=1)

tensor([[-0.2553, -1.4904],
        [-0.9163, -0.5108],
        [-0.7160, -0.6708],
        [-3.1395, -0.0443],
        [-0.7592, -0.6312],
        [-3.0460, -0.0487]])

Let's compute the mean of cross entropy losses across the training examples:

In [9]:
-1*torch.log(p_correct_class), (-1*torch.log(p_correct_class)).mean()

(tensor([0.2553, 0.5108, 0.7160, 0.0443, 0.6312, 3.0460]), tensor(0.8673))

We can just use Pytorch to compute this directly

In [10]:
nn.CrossEntropyLoss(reduction='none')(activations, targets), nn.CrossEntropyLoss()(activations, targets)

(tensor([0.2553, 0.5108, 0.7160, 0.0443, 0.6312, 3.0460]), tensor(0.8673))

or by using:

In [11]:
F.cross_entropy(activations, targets, reduction='none'), F.cross_entropy(activations, targets)

(tensor([0.2553, 0.5108, 0.7160, 0.0443, 0.6312, 3.0460]), tensor(0.8673))

# Gradient of Cross Entropy

We follow {% cite markusthill_ce_note %}

Let $z^{[1]},\ldots, z^{[K]}$ denote the activations corresponding to the $K$ classes. The softmax activation for each class is given by:

$$o^{[j]} = \frac{e^{z^{[j]}}}{\sum_{l=1}^{K} e^{z^{[l]}}}$$

The cross-entropy loss across the $K$ classes is given by: 

$$E=-\sum_{l=1}^{K}y^{[l]}ln(o^{[l]})$$

## Partial derivative of $o^{[j]}$ with respect to $z^{[i]}$

$$\frac{\partial}{\partial z^{[i]}} o^{[j]} = \frac{\partial}{\partial z^{[i]}} \frac{e^{z^{[j]}}}{\sum_l e^{z^{[l]}}}$$
$$= e^{z^{[j]}} \frac{\partial}{\partial z^{[i]}} \Bigg(\sum_l e^{z^{[l]}} \Bigg)^{-1}$$
$$= -e^{z^{[j]}} \Bigg(\sum_l e^{z^{[l]}} \Bigg)^{-2} e^{z^{[i]}} = -o^{[j]} \cdot o^{[i]}$$

## Partial derivative of $o^{[i]}$ with respect to $z^{[i]}$

\begin{align}
  \frac{\partial}{\partial z^{[i]}} o^{[i]} &= \frac{\partial}{\partial z^{[i]}} \frac{e^{z^{[i]}}}{\sum_l e^{z^{[l]}}}\\
  &= \frac{e^{z^{[i]}}}{\sum_{l} e^{z^{[l]}}} + e^{z^{[i]}} \frac{\partial}{\partial z^{[i]}} \Bigg(\sum_l e^{z^{[l]}} \Bigg)^{-1}\\
  &= o^{[i]}-e^{z^{[i]}} \Bigg(\sum_l e^{z^{[l]}} \Bigg)^{-2} e^{z^{[i]}}\\
  &= o^{[i]} - o^{[i]} \cdot o^{[i]} \\
  &= o^{[i]} \cdot (1 - o^{[i]})
  \label{eq:partialOi}
\end{align}

Let's compute the gradient of the cross-entropy loss with respect to the activation of the $i$-the class:

\begin{aligned}
\frac{\partial E}{\partial z^{[i]}}&=- \sum_{l=1}^{K} y^{[l]}\cdot  \frac{\partial}{\partial z^{[i]}}ln(o^{[l]})\\
&=- \sum_{j \neq i} y^{[j]}\cdot  \frac{\partial}{\partial z^{[i]}}ln(o^{[j]}) - y^{[i]}\cdot  \frac{\partial}{\partial z^{[i]}}ln(o^{[i]})\\
&=- \sum_{j \neq i} y^{[j]}\cdot \frac{1}{o^{[j]}} \cdot \frac{\partial}{\partial z^{[i]}}o^{[j]} - y^{[i]}\cdot \frac{1}{o^{[i]}} \cdot \frac{\partial}{\partial z^{[i]}}o^{[i]}\\
&= - \sum_{j \neq i} y^{[j]}\cdot \frac{1}{o^{[j]}} \cdot (-o^{[j]} \cdot o^{[i]}) - y^{[i]}\cdot \frac{1}{o^{[i]}} \cdot o^{[i]} \cdot (1 - o^{[i]})\\
&= - \sum_{j \neq i} y^{[j]} (-o^{[i]}) - y^{[i]} (1 - o^{[i]}) \\
&= - \sum_{j \neq i} y^{[j]} (-o^{[i]}) + y^{[i]} o^{[i]} - y^{[i]} \\
&= o^{[i]} \sum_{j} y^{[j]}  - y^{[i]} \\
&= o^{[i]} - y^{[i]}
\end{aligned}


Per _Sylvain says_ in page 203 of Chapter 5 of {% cite fastbook2020 %}, " _The gradient is proportional to the difference between the prediction and the target._... _Because the gradient is linear we won't see sudden jumps or exponential increases in gradients, which should lead to smoother training of models._"

# References
{% bibliography --cited %}